In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from: https://ollama.com/download  \n   - macOS: install the `.pkg`\n   - Windows: install the `.msi`\n   - Linux: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Open a terminal and run:\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.\n\n3. To test that the local server is running, use:\n   ```bash\n   curl http://localhost:11434\n   ```\n\nIf you want to use it from Python, install the client with:\n```bash\npip install ollama\n```'

In [5]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [6]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [7]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—usually you can join a course after discovering it, **if enrollment is still open**.\n\nA few things to check:\n- **Enrollment deadline**: some courses allow late joining, others don’t.\n- **Prerequisites**: you may need to meet certain requirements first.\n- **Capacity**: if the course is full, you may need to join a waitlist.\n- **Access method**: online courses often let you enroll instantly; in-person courses may require approval.\n\nIf you want, I can help you figure out the exact next step if you tell me:\n1. the **course name/platform**, or  \n2. what kind of course it is (**school, online platform, workshop, etc.**).'

In [8]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join it enrollment registration late join"}', call_id='call_vF6VZ2YrfqNDXoNqDcmIUdW7', name='search', type='function_call', id='fc_03f3d5473cb50de7006a2fab02e8308195bc912ae9fdd1b144', namespace=None, status='completed')]

In [9]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [10]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [11]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.'

In [12]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(652, 31)

In [13]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [14]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [15]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [16]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course late enrollment discovered course can I join"}
function_call: search {"query":"course enrollment discover course can I still join"}
function_call: search {"query":"late join registration course FAQ"}


In [17]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [18]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama local run install local FAQ"}
function_call: search {"query":"Ollama locally run model FAQ"}
function_call: search {"query":"Ollama local setup FAQ"}
iteration #2...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: install the `.pkg`
     - **Windows**: install the `.msi`
     - **Linux**:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a model locally**
   Run:
   ```bash
   ollama run llama3
   ```
   This will download the model the first time and then open a local chat session.

3. **Check that the local server is running**
   In another terminal, try:
   ```bash
   curl http://localhost:11434
   ```
   If Ollama is running, you should get a response from the local server.

4. **Use it from Python**
   Install the client:
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import 

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - Choose your OS:\n     - **macOS**: install the `.pkg`\n     - **Windows**: install the `.msi`\n     - **Linux**:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Start a model locally**\n   Run:\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model the first time and then open a local chat session.\n\n3. **Check that the local server is running**\n   In another terminal, try:\n   ```bash\n   curl http://localhost:11434\n   ```\n   If Ollama is running, you should get a response from the local server.\n\n4. **Use it from Python**\n   Install the client:\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get 

In [19]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enroll discovered course can I still join"}
function_call: search {"query":"late registration enrollment course discovered course join"}
function_call: search {"query":"can I still join the course after it started"}
iteration #2...
ASSISTANT:
Yes — you can still join.

If you want a certificate, make sure you submit your project while submissions are still open. Also, the course can be followed without formal registration, so you can start learning right away.

If you want, I can also help with what to do next to get started.


'Yes — you can still join.\n\nIf you want a certificate, make sure you submit your project while submissions are still open. Also, the course can be followed without formal registration, so you can start learning right away.\n\nIf you want, I can also help with what to do next to get started.'

In [20]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...


function_call: search {"query":"queen gambit chess opening queen's gambit"}
function_call: search {"query":"queen gambit meaning in chess opening"}
function_call: search {"query":"what is the queen's gambit"}
iteration #2...
ASSISTANT:
The **Queen’s Gambit** is a very common **chess opening**:

- **1. d4 d5**
- **2. c4**

White offers the **c-pawn** as a “gambit” to try to pull Black’s **d-pawn** away from the center, so White can gain space and better control of the board.

### Why it’s called a gambit
A **gambit** is when a player offers material, usually a pawn, for some positional advantage.

### Main idea
White wants to:
- control the center,
- create pressure on Black’s d5 pawn,
- and develop pieces smoothly.

### Two main types
- **Queen’s Gambit Accepted**: Black takes the c-pawn.
- **Queen’s Gambit Declined**: Black does not take it.

If you meant the **Netflix show “The Queen’s Gambit,”** that’s about a chess prodigy, but the title comes from this opening.

Want me to show yo

'The **Queen’s Gambit** is a very common **chess opening**:\n\n- **1. d4 d5**\n- **2. c4**\n\nWhite offers the **c-pawn** as a “gambit” to try to pull Black’s **d-pawn** away from the center, so White can gain space and better control of the board.\n\n### Why it’s called a gambit\nA **gambit** is when a player offers material, usually a pawn, for some positional advantage.\n\n### Main idea\nWhite wants to:\n- control the center,\n- create pressure on Black’s d5 pawn,\n- and develop pieces smoothly.\n\n### Two main types\n- **Queen’s Gambit Accepted**: Black takes the c-pawn.\n- **Queen’s Gambit Declined**: Black does not take it.\n\nIf you meant the **Netflix show “The Queen’s Gambit,”** that’s about a chess prodigy, but the title comes from this opening.\n\nWant me to show you the basic moves and ideas in a simple diagram?'

In [21]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit opening chess"}
iteration #3...
ASSISTANT:
I couldn’t find a course-related FAQ entry for “queen gambit,” so I’m not able to answer that from the course materials.

If you meant something else in the course context, feel free to rephrase it. Are there other areas you want to explore?


'I couldn’t find a course-related FAQ entry for “queen gambit,” so I’m not able to answer that from the course materials.\n\nIf you meant something else in the course context, feel free to rephrase it. Are there other areas you want to explore?'

In [22]:
!uv add toyaikit

Resolved 127 packages in 4.13s                                       
⠙ Preparing packages... (0/7)                                                   ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/7)-------------------     0 B/21.96 KiB           
⠙ Preparing packages... (0/7)---------- 14.80 KiB/21.96 KiB         
⠙ Preparing packages... (0/7)---------- 14.80 KiB/21.96 KiB         
docstring-parser     ------------------------------ 14.80 KiB/21.96 KiB
⠙ Preparing packages... (0/7)-------------------     0 B/902.20 KiB          
docstring-parser     ------------------------------ 14.80 KiB/21.96 KiB
⠙ Preparing packages... (0/7)------------------- 14.84 KiB/902.20 KiB        
docstring-parser     ------------------------------ 14.80 KiB/21.96 KiB
⠙ Preparing packages... (0/7)------------------- 30.84 KiB/902.20 KiB        
docstring-parser     ------------------------------ 14.80 KiB/21.96 KiB
⠙ Preparing packages... (0/7)--

In [23]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [24]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [25]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [26]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [27]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [28]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [30]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [31]:
result.cost

CostInfo(input_cost=Decimal('0.00264225'), output_cost=Decimal('0.0014535'), total_cost=Decimal('0.00409575'))

In [32]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama run locally Ollama locally installation local run"}', ca

In [33]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [34]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='how do you do!', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"how do you do greeting course FAQ"}', call_id='call_l